# Comprehensive COVID-19 Pandemic Data Analysis

A reproducible analysis notebook built on reusable project modules.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR
from src.data_loader import (
    add_daily_changes,
    aggregate_country_daily,
    clean_covid_data,
    discover_csv_files,
    duplicate_report,
    get_latest_country_data,
    inspect_schema,
    load_covid_data,
    missing_data_report,
    remove_exact_duplicates,
)
from src.visualization import plot_country_trends, plot_global_trends, plot_top_countries


## 1. Schema and Report-Date Inspection
Inspect source files before loading the complete dataset.


In [ ]:
csv_files = discover_csv_files(DATA_DIR)
schema_report = inspect_schema(csv_files)
date_report = pd.DataFrame({
    'Source_File': list(schema_report['dates_from_filenames']),
    'Extracted_Date': list(schema_report['dates_from_filenames'].values()),
})
display(date_report)


## 2. Load and Standardize Data


In [ ]:
df = load_covid_data(DATA_DIR)
print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')
display(df.head())


## 3. Missing Data


In [ ]:
missing_before = missing_data_report(df)
display(missing_before)
df = clean_covid_data(df)
display(missing_data_report(df))


## 4. Duplicate Analysis


In [ ]:
print(duplicate_report(df))
df = remove_exact_duplicates(df)
print(duplicate_report(df))


## 5. Country-Level Daily Time Series


In [ ]:
country_daily = add_daily_changes(aggregate_country_daily(df))
display(country_daily.head())


## 6. Global Time Series and Visualizations


In [ ]:
global_daily = country_daily.groupby('Date', as_index=False)[[c for c in ['Confirmed', 'Deaths', 'Recovered', 'Active'] if c in country_daily.columns]].sum(min_count=1)
display(global_daily.tail())
plot_global_trends(global_daily)


## 7. Latest Available Data by Country


In [ ]:
latest_country_data = get_latest_country_data(df)
display(latest_country_data.head(20))


## 8. Country Visualization Example
Replace the example country with a country present in your dataset.


In [ ]:
example_country = latest_country_data['Country_Region'].dropna().iloc[0]
plot_country_trends(country_daily, example_country)
plot_top_countries(latest_country_data, metric='Confirmed', top_n=10)


## 9. Final Data Quality Summary


In [ ]:
print(f'Final record count: {len(df):,}')
print(f'Country-date records: {len(country_daily):,}')
print(f'Countries: {country_daily["Country_Region"].nunique()}')
print(f'Date range: {country_daily["Date"].min()} to {country_daily["Date"].max()}')
